In [ ]:
#CONNECT GPU

MODULE 1

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install datasets transformers torch accelerate -q

In [ ]:
import os
import random
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import RobertaTokenizer
import json
from collections import defaultdict

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✓ Libraries imported and seed set")

✓ Libraries imported and seed set


In [ ]:
# Directory in Google Drive to save datasets
SAVE_DIR = '/content/drive/MyDrive/FYP_GradientFree_LoRA/'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'datasets'), exist_ok=True)

print(f"✓ Datasets will be saved to: {SAVE_DIR}")

✓ Datasets will be saved to: /content/drive/MyDrive/FYP_GradientFree_LoRA/


In [ ]:
# Patterns and verbalizers for each dataset
DATASET_CONFIG = {
    'sst2': {
        'dataset_name': 'sst2',
        'dataset_config': None,
        'text_column': 'sentence',
        'label_column': 'label',
        'pattern': lambda text: f"{text} It was <mask>.",
        'verbalizers': {0: "bad", 1: "great"},
        'num_classes': 2,
        'use_validation_as_test': True  # SST-2 test set has no labels
    },
    'yelp_polarity': {
        'dataset_name': 'yelp_polarity',
        'dataset_config': None,
        'text_column': 'text',
        'label_column': 'label',
        'pattern': lambda text: f"{text} It was <mask>.",
        'verbalizers': {0: "bad", 1: "great"},
        'num_classes': 2,
        'use_validation_as_test': False
    },
    'ag_news': {
        'dataset_name': 'ag_news',
        'dataset_config': None,
        'text_column': 'text',
        'label_column': 'label',
        'pattern': lambda text: f"<mask> News {text}.",
        'verbalizers': {0: "World", 1: "Sports", 2: "Business", 3: "Tech"},
        'num_classes': 4,
        'use_validation_as_test': False
    },
    'dbpedia_14': {
        'dataset_name': 'dbpedia_14',
        'dataset_config': None,
        'text_column': 'content',
        'label_column': 'label',
        'pattern': lambda text: f"Category: <mask> {text}",
        'verbalizers': {
            0: "Company", 1: "Education", 2: "Artist", 3: "Athlete",
            4: "Office", 5: "Transportation", 6: "Building", 7: "Natural",
            8: "Village", 9: "Animal", 10: "Plant", 11: "Album",
            12: "Film", 13: "Written"
        },
        'num_classes': 14,
        'use_validation_as_test': False
    }
}

print("✓ Dataset configurations defined")
print(f"Datasets to process: {list(DATASET_CONFIG.keys())}")

✓ Dataset configurations defined
Datasets to process: ['sst2', 'yelp_polarity', 'ag_news', 'dbpedia_14']


In [ ]:
# Initializing RoBERTa tokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
print("✓ RoBERTa tokenizer loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✓ RoBERTa tokenizer loaded


In [ ]:
def create_few_shot_split(dataset, n_samples_per_class, num_classes, label_column, seed=42):

    random.seed(seed)
    np.random.seed(seed)

    # Group samples by class
    class_samples = defaultdict(list)
    for idx, example in enumerate(dataset):
        label = example[label_column]
        class_samples[label].append(idx)

    # Sample n instances per class for train and dev
    train_indices = []
    dev_indices = []

    for label in range(num_classes):
        if label not in class_samples:
            print(f"Warning: No samples found for class {label}")
            continue

        samples = class_samples[label]
        random.shuffle(samples)


        if len(samples) < 2 * n_samples_per_class:
            print(f"Warning: Class {label} has only {len(samples)} samples, need {2*n_samples_per_class}")
            n = min(n_samples_per_class, len(samples) // 2)
        else:
            n = n_samples_per_class

        train_indices.extend(samples[:n])
        dev_indices.extend(samples[n:2*n])

    train_split = dataset.select(train_indices)
    dev_split = dataset.select(dev_indices)

    return train_split, dev_split

print("✓ Few-shot split function defined")

✓ Few-shot split function defined


In [ ]:
def apply_pattern_and_verbalizer(example, config):
    text = example[config['text_column']]
    label = example[config['label_column']]

    max_text_length = 400
    if len(text) > max_text_length:
        text = text[:max_text_length]

    # Apply pattern
    mlm_text = config['pattern'](text)

    # Getting verbalizer for the label
    verbalizer = config['verbalizers'][label]

    return {
        'mlm_text': mlm_text,
        'text': text,
        'label': label,
        'verbalizer': verbalizer
    }

print("✓ Pattern application function defined")

✓ Pattern application function defined


In [ ]:
def process_dataset(dataset_key, n_shot=16, save=True):
    """
    Process a single dataset: load, create few-shot splits, apply patterns
    """
    print(f"\n{'='*60}")
    print(f"Processing {dataset_key} with {n_shot}-shot setting")
    print(f"{'='*60}")

    config = DATASET_CONFIG[dataset_key]

    # Loading dataset
    print("Loading dataset")
    if config['dataset_config']:
        dataset = load_dataset(config['dataset_name'], config['dataset_config'])
    else:
        dataset = load_dataset(config['dataset_name'])


    if config.get('use_validation_as_test', False):
        print("Using validation set as test set (original test set has no labels)")
        train_full = dataset['train']
        test_full = dataset['validation']
    else:
        train_full = dataset['train']
        test_full = dataset['test']


    valid_labels = set(config['verbalizers'].keys())
    print(f"Valid labels: {valid_labels}")

    train_full = train_full.filter(lambda x: x[config['label_column']] in valid_labels)
    test_full = test_full.filter(lambda x: x[config['label_column']] in valid_labels)

    print(f"Original train size (after filtering): {len(train_full)}")
    print(f"Original test size (after filtering): {len(test_full)}")

    # Creating few-shot splits
    print(f"\nCreating {n_shot}-shot splits...")
    train_split, dev_split = create_few_shot_split(
        train_full,
        n_samples_per_class=n_shot,
        num_classes=config['num_classes'],
        label_column=config['label_column'],
        seed=42
    )

    print(f"Few-shot train size: {len(train_split)}")
    print(f"Few-shot dev size: {len(dev_split)}")

    # Apply patterns and verbalizers
    print("\nApplying patterns and verbalizers...")
    train_processed = train_split.map(
        lambda x: apply_pattern_and_verbalizer(x, config),
        batched=False
    )
    dev_processed = dev_split.map(
        lambda x: apply_pattern_and_verbalizer(x, config),
        batched=False
    )
    test_processed = test_full.map(
        lambda x: apply_pattern_and_verbalizer(x, config),
        batched=False
    )

    # Create DatasetDict
    processed_dataset = DatasetDict({
        'train': train_processed,
        'validation': dev_processed,
        'test': test_processed
    })


    print("\n--- Sample from training set ---")
    print(f"Original text: {train_processed[0]['text'][:100]}...")
    print(f"MLM text: {train_processed[0]['mlm_text'][:150]}...")
    print(f"Label: {train_processed[0]['label']}")
    print(f"Verbalizer: {train_processed[0]['verbalizer']}")

    # Save to Google Drive
    if save:
        save_path = os.path.join(SAVE_DIR, 'datasets', f'{dataset_key}_{n_shot}shot')
        print(f"\nSaving to: {save_path}")
        processed_dataset.save_to_disk(save_path)


        config_save = {
            'pattern': str(config['pattern']('TEXT_PLACEHOLDER')),
            'verbalizers': config['verbalizers'],
            'n_shot': n_shot,
            'num_classes': config['num_classes']
        }
        with open(os.path.join(save_path, 'config.json'), 'w') as f:
            json.dump(config_save, f, indent=2)

    print(f"\n✓ {dataset_key} processing complete!")

    return processed_dataset

print("✓ Main processing function defined")

✓ Main processing function defined


In [ ]:
# Setting the number of samples per class for few-shot learning
n_shot = 16

print(f"✓ Using {n_shot}-shot setting")

✓ Using 16-shot setting


In [ ]:
# Process SST-2 (Sentiment Analysis)
print("\n" + "="*60)
print("PROCESSING SST-2")
print("="*60)

sst2_dataset = process_dataset('sst2', n_shot=n_shot)


PROCESSING SST-2

Processing sst2 with 16-shot setting
Loading dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

... (10 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/872 [00:00<?, ? examples/s]


✓ sst2 processing complete!


In [ ]:
# Process Yelp Polarity (Sentiment Analysis)
print("\n" + "="*60)
print("PROCESSING YELP POLARITY")
print("="*60)

yelp_dataset = process_dataset('yelp_polarity', n_shot=n_shot)


PROCESSING YELP POLARITY

Processing yelp_polarity with 16-shot setting
Loading dataset


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

... (8 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/38000 [00:00<?, ? examples/s]


✓ yelp_polarity processing complete!


In [ ]:
# Process AG News (Topic Classification)
print("\n" + "="*60)
print("PROCESSING AG NEWS")
print("="*60)

agnews_dataset = process_dataset('ag_news', n_shot=n_shot)


PROCESSING AG NEWS

Processing ag_news with 16-shot setting
Loading dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

... (8 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/64 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/64 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]


✓ ag_news processing complete!


In [ ]:
# Process DBPedia (Topic Classification)
print("\n" + "="*60)
print("PROCESSING DBPEDIA")
print("="*60)

dbpedia_dataset = process_dataset('dbpedia_14', n_shot=n_shot)


PROCESSING DBPEDIA

Processing dbpedia_14 with 16-shot setting
Loading dataset


README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

... (8 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/70000 [00:00<?, ? examples/s]


✓ dbpedia_14 processing complete!


In [ ]:
# Print summary of all processed datasets
print("\n" + "="*60)
print("PROCESSING COMPLETE - SUMMARY")
print("="*60)

processed_datasets = {
    'sst2': sst2_dataset,
    'yelp_polarity': yelp_dataset,
    'ag_news': agnews_dataset,
    'dbpedia_14': dbpedia_dataset
}

for name, ds in processed_datasets.items():
    print(f"\n{name.upper()}:")
    print(f"  Train: {len(ds['train'])} samples")
    print(f"  Validation: {len(ds['validation'])} samples")
    print(f"  Test: {len(ds['test'])} samples")

print(f"\n✓ All datasets saved to: {SAVE_DIR}datasets/")
print("\n--- How to load datasets later ---")
print("from datasets import load_from_disk")
print("dataset = load_from_disk('/content/drive/MyDrive/FYP_GradientFree_LoRA/datasets/sst2_16shot')")


PROCESSING COMPLETE - SUMMARY

SST2:
  Train: 32 samples
  Validation: 32 samples
  Test: 872 samples

YELP_POLARITY:
  Train: 32 samples
  Validation: 32 samples
  Test: 38000 samples

AG_NEWS:
  Train: 64 samples
  Validation: 64 samples
  Test: 7600 samples

DBPEDIA_14:
  Train: 224 samples
  Validation: 224 samples
  Test: 70000 samples

✓ All datasets saved to: /content/drive/MyDrive/FYP_GradientFree_LoRA/datasets/

--- How to load datasets later ---
from datasets import load_from_disk
dataset = load_from_disk('/content/drive/MyDrive/FYP_GradientFree_LoRA/datasets/sst2_16shot')


In [ ]:
import pandas as pd
from IPython.display import display


def display_dataset_as_table(dataset_name, dataset, split='train', max_rows=None):
    print(f"\n{'='*100}")
    print(f"{dataset_name.upper()} - {split.upper()} SPLIT")
    print(f"{'='*100}\n")


    df = dataset[split].to_pandas()


    df['text_preview'] = df['text'].str[:100] + '...'
    df['mlm_text_preview'] = df['mlm_text'].str[:150] + '...'


    display_cols = ['label', 'verbalizer', 'text_preview', 'mlm_text_preview']
    df_display = df[display_cols]


    pd.set_option('display.max_rows', max_rows)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)

    print(f"Total records: {len(df)}\n")
    display(df_display)

    return df

# Display all datasets

# SST-2
sst2_train_df = display_dataset_as_table('sst2', sst2_dataset, 'train', max_rows=None)
sst2_val_df = display_dataset_as_table('sst2', sst2_dataset, 'validation', max_rows=None)
sst2_test_df = display_dataset_as_table('sst2', sst2_dataset, 'test', max_rows=50)  # Limit test to 50 rows

# Yelp Polarity
yelp_train_df = display_dataset_as_table('yelp_polarity', yelp_dataset, 'train', max_rows=None)
yelp_val_df = display_dataset_as_table('yelp_polarity', yelp_dataset, 'validation', max_rows=None)
yelp_test_df = display_dataset_as_table('yelp_polarity', yelp_dataset, 'test', max_rows=50)

# AG News
agnews_train_df = display_dataset_as_table('ag_news', agnews_dataset, 'train', max_rows=None)
agnews_val_df = display_dataset_as_table('ag_news', agnews_dataset, 'validation', max_rows=None)
agnews_test_df = display_dataset_as_table('ag_news', agnews_dataset, 'test', max_rows=50)

# DBPedia
dbpedia_train_df = display_dataset_as_table('dbpedia_14', dbpedia_dataset, 'train', max_rows=None)
dbpedia_val_df = display_dataset_as_table('dbpedia_14', dbpedia_dataset, 'validation', max_rows=None)
dbpedia_test_df = display_dataset_as_table('dbpedia_14', dbpedia_dataset, 'test', max_rows=50)

print("\n" + "="*100)
print("✓ All datasets displayed!")
print("="*100)


SST2 - TRAIN SPLIT

Total records: 32



,label,verbalizer,text_preview,mlm_text_preview
0,0,bad,exactly how genteel and unsurprising the execution turns out to be ...,exactly how genteel and unsurprising the execution turns out to be It was <mask>....
1,0,bad,an all-time low for kevin costner ...,an all-time low for kevin costner It was <mask>....
2,0,bad,several cliched movie structures : ...,several cliched movie structures : It was <mask>....
3,0,bad,america 's knee-jerk moral sanctimony ...,america 's knee-jerk moral sanctimony It was <mask>....
4,0,bad,a dumb comedy ...,a dumb comedy It was <mask>....
5,0,bad,the plan to make enough into ` an inspiring tale of survival wrapped in the heart-pounding suspense ...,the plan to make enough into ` an inspiring tale of survival wrapped in the heart-pounding suspense of a stylish psychological thriller ' has flopped ...
6,0,bad,"proficient , dull ...","proficient , dull It was <mask>...."
7,0,bad,chitchat that only self-aware neurotics engage in ...,chitchat that only self-aware neurotics engage in It was <mask>....
8,0,bad,tries to cram too many ingredients into one small pot . ...,tries to cram too many ingredients into one small pot . It was <mask>....
9,0,bad,paper-thin characterizations ...,paper-thin characterizations It was <mask>....



SST2 - VALIDATION SPLIT

Total records: 32



,label,verbalizer,text_preview,mlm_text_preview
0,0,bad,bad imitation ...,bad imitation It was <mask>....
1,0,bad,"listless , witless , ...","listless , witless , It was <mask>...."
2,0,bad,gussied up with so many distracting special effects and visual party tricks that it 's not clear whe...,gussied up with so many distracting special effects and visual party tricks that it 's not clear whether we 're supposed to shriek It was <mask>....
3,0,bad,ca n't disguise the fact that it 's inauthentic at its core and that its story just is n't worth tel...,ca n't disguise the fact that it 's inauthentic at its core and that its story just is n't worth telling . It was <mask>....
4,0,bad,put through torture ...,put through torture It was <mask>....
5,0,bad,uncomfortably strained ...,uncomfortably strained It was <mask>....
6,0,bad,is that they 're stuck with a script that prevents them from firing on all cylinders . ...,is that they 're stuck with a script that prevents them from firing on all cylinders . It was <mask>....
7,0,bad,wobbly ...,wobbly It was <mask>....
8,0,bad,sure to ultimately disappoint the action fans who will be moved to the edge of their seats by the dy...,sure to ultimately disappoint the action fans who will be moved to the edge of their seats by the dynamic first act It was <mask>....
9,0,bad,freewheeling trash-cinema roots ...,freewheeling trash-cinema roots It was <mask>....



SST2 - TEST SPLIT

Total records: 872



,label,verbalizer,text_preview,mlm_text_preview
0,1,great,it 's a charming and often affecting journey . ...,it 's a charming and often affecting journey . It was <mask>....
1,0,bad,unflinchingly bleak and desperate ...,unflinchingly bleak and desperate It was <mask>....
2,1,great,allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmma...,allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker . It was <mask>....
3,1,great,"the acting , costumes , music , cinematography and sound are all astounding given the production 's ...","the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales . It was <mask>...."
4,0,bad,"it 's slow -- very , very slow . ...","it 's slow -- very , very slow . It was <mask>...."
...,...,...,...,...
867,0,bad,has all the depth of a wading pool . ...,has all the depth of a wading pool . It was <mask>....
868,1,great,a movie with a real anarchic flair . ...,a movie with a real anarchic flair . It was <mask>....
869,0,bad,a subject like this should inspire reaction in its audience ; the pianist does not . ...,a subject like this should inspire reaction in its audience ; the pianist does not . It was <mask>....
870,0,bad,... is an arthritic attempt at directing by callie khouri . ...,... is an arthritic attempt at directing by callie khouri . It was <mask>....


... (15 outputs left out)

,label,verbalizer,text_preview,mlm_text_preview
0,0,Company,International Tea Co. Stores was a leading chain of grocers based in London. It was a constituent o...,Category: <mask> International Tea Co. Stores was a leading chain of grocers based in London. It was a constituent of the FT 30 index of leading comp...
1,0,Company,First Transit is a United States based subsidiary of FirstGroup. It provides contract public transi...,Category: <mask> First Transit is a United States based subsidiary of FirstGroup. It provides contract public transit and paratransit services transi...
2,0,Company,Shepherd and Wedderburn LLP is a leading commercial law firm headquartered in Edinburgh and with of...,Category: <mask> Shepherd and Wedderburn LLP is a leading commercial law firm headquartered in Edinburgh and with offices in Glasgow Aberdeen and Lon...
3,0,Company,Hulman & Company is a private family-owned company founded in 1850 by Francis Hulman as a wholesale...,Category: <mask> Hulman & Company is a private family-owned company founded in 1850 by Francis Hulman as a wholesale grocery tobacco and liquor store...
4,0,Company,Brashs was one of Australia's biggest music retail chains specializing in the retail of various vin...,Category: <mask> Brashs was one of Australia's biggest music retail chains specializing in the retail of various vinyl records and CDs blank audio ca...
5,0,Company,Science Applications International Corporation (SAIC) is a US company headquartered in McLean Virgi...,Category: <mask> Science Applications International Corporation (SAIC) is a US company headquartered in McLean Virginia that provides government serv...
6,0,Company,The Open Geospatial Consortium (OGC) an international voluntary consensus standards organization or...,Category: <mask> The Open Geospatial Consortium (OGC) an international voluntary consensus standards organization originated in 1994. In the OGC more...
7,0,Company,AET Tanker Holdings Sdn Bhd formerly American Eagle Tankers is a Malaysian-owned global shipping co...,Category: <mask> AET Tanker Holdings Sdn Bhd formerly American Eagle Tankers is a Malaysian-owned global shipping company specialising in the ocean t...
8,0,Company,InterCity Group (NZ) Ltd is a passenger transport and tourism company and operates New Zealand's la...,Category: <mask> InterCity Group (NZ) Ltd is a passenger transport and tourism company and operates New Zealand's largest coach network and ferries a...
9,0,Company,TechnoMarine was founded in 1997 by French entrepreneur Franck Dubarry in Geneva with the launch of...,Category: <mask> TechnoMarine was founded in 1997 by French entrepreneur Franck Dubarry in Geneva with the launch of the Raft a chronograph on a tran...



DBPEDIA_14 - TEST SPLIT

Total records: 70000



,label,verbalizer,text_preview,mlm_text_preview
0,0,Company,TY KU /taɪkuː/ is an American alcoholic beverage company that specializes in sake and other spirits...,Category: <mask> TY KU /taɪkuː/ is an American alcoholic beverage company that specializes in sake and other spirits. The privately-held company was ...
1,0,Company,OddLot Entertainment founded in 2001 by longtime producers Gigi Pritzker and Deborah Del Prete (The...,Category: <mask> OddLot Entertainment founded in 2001 by longtime producers Gigi Pritzker and Deborah Del Prete (The Wedding Planner) is a film produ...
2,0,Company,Henkel AG & Company KGaA operates worldwide with leading brands and technologies in three business ...,Category: <mask> Henkel AG & Company KGaA operates worldwide with leading brands and technologies in three business areas: Laundry & Home Care Beauty...
3,0,Company,The GOAT Store (Games Of All Type Store) LLC is one of the largest retro gaming online stores and a...,Category: <mask> The GOAT Store (Games Of All Type Store) LLC is one of the largest retro gaming online stores and an Independent Video Game Publishi...
4,0,Company,RagWing Aircraft Designs (also called the RagWing Aeroplane Company and RagWing Aviation) was an Am...,Category: <mask> RagWing Aircraft Designs (also called the RagWing Aeroplane Company and RagWing Aviation) was an American aircraft design and manufa...
...,...,...,...,...
69995,13,Written,Energy Victory: Winning the War on Terror by Breaking Free of Oil is a 2007 book by Robert Zubrin. ...,Category: <mask> Energy Victory: Winning the War on Terror by Breaking Free of Oil is a 2007 book by Robert Zubrin. Zubrin's central argument is that...
69996,13,Written,Bestiario is a book of 8 short stories written by Julio Cortázar....,Category: <mask> Bestiario is a book of 8 short stories written by Julio Cortázar....
69997,13,Written,Wuthering Heights is a novel by Emily Brontë written between October 1845 and June 1846 and publish...,Category: <mask> Wuthering Heights is a novel by Emily Brontë written between October 1845 and June 1846 and published in 1847 under the pseudonym El...
69998,13,Written,L'Indépendant is a newspaper published in Luxembourg from 1945....,Category: <mask> L'Indépendant is a newspaper published in Luxembourg from 1945....



✓ All datasets displayed!


In [ ]:
def tokenize_and_prepare_for_mlm(examples, tokenizer, config, max_length=256):

    tokenized = tokenizer(
        examples['mlm_text'],
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors=None
    )

    # Finding mask token position and create verbalizer token ids
    mask_token_id = tokenizer.mask_token_id
    verbalizer_ids = []
    mask_positions = []

    for i, input_ids in enumerate(tokenized['input_ids']):

        try:
            mask_pos = input_ids.index(mask_token_id)
            mask_positions.append(mask_pos)
        except ValueError:

            mask_positions.append(-1)


        label = examples['label'][i]
        verbalizer_text = config['verbalizers'][label]


        verbalizer_tokens = tokenizer(verbalizer_text, add_special_tokens=False)['input_ids']


        if len(verbalizer_tokens) > 0:
            verbalizer_ids.append(verbalizer_tokens[0])
        else:
            verbalizer_ids.append(-100)


    tokenized['mask_position'] = mask_positions
    tokenized['verbalizer_id'] = verbalizer_ids
    tokenized['labels'] = examples['label']

    return tokenized


print("✓ Tokenization function defined")

✓ Tokenization function defined


In [ ]:
print("="*70)
print("TOKENIZING DATASETS")
print("="*70)


MAX_LENGTH = 256

# Store tokenized datasets
tokenized_datasets = {}

datasets_to_tokenize = {
    'sst2': sst2_dataset,
    'yelp_polarity': yelp_dataset,
    'ag_news': agnews_dataset,
    'dbpedia_14': dbpedia_dataset
}

for dataset_name, dataset in datasets_to_tokenize.items():
    print(f"\n{'─'*70}")
    print(f"Tokenizing {dataset_name.upper()}")
    print(f"{'─'*70}")

    config = DATASET_CONFIG[dataset_name]

    # Tokenize all splits
    tokenized_train = dataset['train'].map(
        lambda examples: tokenize_and_prepare_for_mlm(examples, tokenizer, config, MAX_LENGTH),
        batched=True,
        batch_size=32,
        remove_columns=dataset['train'].column_names  # Remove original text columns
    )

    tokenized_val = dataset['validation'].map(
        lambda examples: tokenize_and_prepare_for_mlm(examples, tokenizer, config, MAX_LENGTH),
        batched=True,
        batch_size=32,
        remove_columns=dataset['validation'].column_names
    )

    tokenized_test = dataset['test'].map(
        lambda examples: tokenize_and_prepare_for_mlm(examples, tokenizer, config, MAX_LENGTH),
        batched=True,
        batch_size=32,
        remove_columns=dataset['test'].column_names
    )


    tokenized_dataset = DatasetDict({
        'train': tokenized_train,
        'validation': tokenized_val,
        'test': tokenized_test
    })

    tokenized_datasets[dataset_name] = tokenized_dataset

    print(f"✓ {dataset_name} tokenized")
    print(f"  Train: {len(tokenized_train)} samples")
    print(f"  Validation: {len(tokenized_val)} samples")
    print(f"  Test: {len(tokenized_test)} samples")
    print(f"  Columns: {tokenized_train.column_names}")

print("\n" + "="*70)
print("✓ All datasets tokenized!")
print("="*70)

TOKENIZING DATASETS

──────────────────────────────────────────────────────────────────────
Tokenizing SST2
──────────────────────────────────────────────────────────────────────


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

✓ sst2 tokenized
  Train: 32 samples
  Validation: 32 samples
  Test: 872 samples
  Columns: ['input_ids', 'attention_mask', 'mask_position', 'verbalizer_id', 'labels']

──────────────────────────────────────────────────────────────────────
Tokenizing YELP_POLARITY
──────────────────────────────────────────────────────────────────────


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

... (7 outputs left out)

Map:   0%|          | 0/224 [00:00<?, ? examples/s]

Map:   0%|          | 0/224 [00:00<?, ? examples/s]

Map:   0%|          | 0/70000 [00:00<?, ? examples/s]

✓ dbpedia_14 tokenized
  Train: 224 samples
  Validation: 224 samples
  Test: 70000 samples
  Columns: ['input_ids', 'attention_mask', 'mask_position', 'verbalizer_id', 'labels']

✓ All datasets tokenized!


In [ ]:
# Inspecting a tokenized sample to verify everything is correct
print("="*70)
print("TOKENIZED DATA INSPECTION")
print("="*70)

for dataset_name, dataset in tokenized_datasets.items():
    print(f"\n{'─'*70}")
    print(f"{dataset_name.upper()} - Sample from Train")
    print(f"{'─'*70}")

    sample = dataset['train'][0]

    print(f"\nInput IDs (first 20): {sample['input_ids'][:20]}")
    print(f"Attention Mask (first 20): {sample['attention_mask'][:20]}")
    print(f"Mask Position: {sample['mask_position']}")
    print(f"Verbalizer ID: {sample['verbalizer_id']}")
    print(f"Label: {sample['labels']}")

    decoded_text = tokenizer.decode(sample['input_ids'])
    print(f"\nDecoded text: {decoded_text}")


    verbalizer_token = tokenizer.decode([sample['verbalizer_id']])
    print(f"Verbalizer token: '{verbalizer_token}'")


    if sample['mask_position'] != -1:
        mask_token = sample['input_ids'][sample['mask_position']]
        print(f"Token at mask position: {tokenizer.decode([mask_token])} (should be <mask>)")

    print()

TOKENIZED DATA INSPECTION

──────────────────────────────────────────────────────────────────────
SST2 - Sample from Train
──────────────────────────────────────────────────────────────────────

Input IDs (first 20): [0, 3463, 45548, 141, 821, 8530, 523, 8, 36637, 21434, 5, 7356, 4072, 66, 7, 28, 1437, 85, 21, 50264]
Attention Mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Mask Position: 19
Verbalizer ID: 10999
Label: 0

Decoded text: <s>exactly how genteel and unsurprising the execution turns out to be  It was<mask>.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><p

In [ ]:
# Save tokenized datasets to Google Drive
print("="*70)
print("SAVING TOKENIZED DATASETS")
print("="*70)

tokenized_save_dir = os.path.join(SAVE_DIR, 'datasets_tokenized')
os.makedirs(tokenized_save_dir, exist_ok=True)

for dataset_name, dataset in tokenized_datasets.items():
    save_path = os.path.join(tokenized_save_dir, f'{dataset_name}_{n_shot}shot_tokenized')
    print(f"\nSaving {dataset_name} to: {save_path}")

    dataset.save_to_disk(save_path)


    tokenizer_info = {
        'tokenizer_name': 'roberta-base',
        'max_length': MAX_LENGTH,
        'vocab_size': tokenizer.vocab_size,
        'mask_token': tokenizer.mask_token,
        'mask_token_id': tokenizer.mask_token_id
    }

    with open(os.path.join(save_path, 'tokenizer_info.json'), 'w') as f:
        json.dump(tokenizer_info, f, indent=2)

    print(f"✓ {dataset_name} saved")

print("\n" + "="*70)
print("✓ All tokenized datasets saved!")
print(f"Location: {tokenized_save_dir}")
print("="*70)

print("\n--- How to load tokenized datasets later ---")
print("from datasets import load_from_disk")
print("dataset = load_from_disk('/content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized/sst2_16shot_tokenized')")

SAVING TOKENIZED DATASETS

Saving sst2 to: /content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized/sst2_16shot_tokenized


Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/872 [00:00<?, ? examples/s]

✓ sst2 saved

Saving yelp_polarity to: /content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized/yelp_polarity_16shot_tokenized


Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

... (7 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/224 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/70000 [00:00<?, ? examples/s]

✓ dbpedia_14 saved

✓ All tokenized datasets saved!
Location: /content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized

--- How to load tokenized datasets later ---
from datasets import load_from_disk
dataset = load_from_disk('/content/drive/MyDrive/FYP_GradientFree_LoRA/datasets_tokenized/sst2_16shot_tokenized')


MODULE 2 & 3

MODULE 4

In [ ]:
from torch.utils.data import DataLoader

def create_dataloaders(tokenized_datasets, batch_size=8):
    """
    Creating DataLoaders for train, validation, and test sets
    """
    def collate_fn(examples):

        batch = {
            'input_ids': torch.tensor([ex['input_ids'] for ex in examples], dtype=torch.long),
            'attention_mask': torch.tensor([ex['attention_mask'] for ex in examples], dtype=torch.long),
            'mask_position': torch.tensor([ex['mask_position'] for ex in examples], dtype=torch.long),
            'verbalizer_id': torch.tensor([ex['verbalizer_id'] for ex in examples], dtype=torch.long),
            'labels': torch.tensor([ex['labels'] for ex in examples], dtype=torch.long)
        }
        return batch

    dataloaders = {}

    for dataset_name, dataset in tokenized_datasets.items():
        dataloaders[dataset_name] = {
            'train': DataLoader(
                dataset['train'],
                batch_size=batch_size,
                shuffle=True,
                collate_fn=collate_fn
            ),
            'validation': DataLoader(
                dataset['validation'],
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_fn
            ),
            'test': DataLoader(
                dataset['test'],
                batch_size=batch_size,
                shuffle=False,
                collate_fn=collate_fn
            )
        }

    return dataloaders


print("✓ DataLoader creation function defined")

✓ DataLoader creation function defined


In [ ]:
from transformers import RobertaForMaskedLM

def evaluate_model_mlm(model_base, lm_head, dataloader, device, dataset_config):
    """
    Evaluate model on MLM task with proper LM head

    Args:
        model_base: RoBERTa base model with LoRA
        lm_head: Language model head for predictions
        dataloader: DataLoader
        device: torch device
        dataset_config: Config with verbalizers

    Returns:
        Dictionary with metrics
    """
    model_base.eval()
    lm_head.eval()

    all_predictions = []
    all_labels = []


    all_verbalizer_ids = []
    label_mapping = {}

    for label, verb_text in dataset_config['verbalizers'].items():
        verb_tokens = tokenizer(verb_text, add_special_tokens=False)['input_ids']
        if len(verb_tokens) > 0:
            verb_id = verb_tokens[0]
            all_verbalizer_ids.append(verb_id)
            label_mapping[verb_id] = label

    print(f"  Verbalizer mapping: {label_mapping}")

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            mask_positions = batch['mask_position']
            labels = batch['labels']

            # Forward pass through base model
            outputs = model_base(
                input_ids=input_ids,
                attention_mask=attention_mask
            )


            hidden_states = outputs.last_hidden_state


            logits = lm_head(hidden_states)


            batch_size = input_ids.size(0)

            for i in range(batch_size):
                mask_pos = mask_positions[i].item()

                if mask_pos != -1:

                    mask_logits = logits[i, mask_pos, :]


                    verbalizer_logits = mask_logits[all_verbalizer_ids]


                    pred_idx = torch.argmax(verbalizer_logits).item()
                    pred_verb_id = all_verbalizer_ids[pred_idx]
                    predicted_label = label_mapping[pred_verb_id]

                    all_predictions.append(predicted_label)
                    all_labels.append(labels[i].item())

    # Compute metrics
    if len(all_predictions) == 0:
        return {
            'accuracy': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'f1': 0.0
        }

    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_predictions, average='weighted', zero_division=0
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


print("✓ FIXED MLM evaluation function defined")

✓ FIXED MLM evaluation function defined


In [ ]:
import torch
import time
import psutil
import json
import os
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import RobertaForMaskedLM
from datasets import load_from_disk

print("✓ Imports ready for Manual Prompt baseline")

✓ Imports ready for Manual Prompt baseline


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
def load_manual_prompt_model(device):
    print("=" * 70)
    print("MANUAL PROMPT — Loading frozen RoBERTa-MLM")
    print("=" * 70)

    model = RobertaForMaskedLM.from_pretrained('roberta-base')

    # Freeze everything — Manual Prompt trains NOTHING
    for param in model.parameters():
        param.requires_grad = False

    model.eval()
    model = model.to(device)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"  Total params     : {total:,}")
    print(f"  Trainable params : {trainable}  ← must be 0")
    print("✓ Frozen RoBERTa-MLM loaded")

    return model


manual_prompt_model = load_manual_prompt_model(device)

MANUAL PROMPT — Loading frozen RoBERTa-MLM


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Total params     : 124,697,433
  Trainable params : 0  ← must be 0
✓ Frozen RoBERTa-MLM loaded


In [ ]:
def evaluate_manual_prompt(full_mlm_model, dataloader, device, dataset_config):
    """
    Splits RobertaForMaskedLM into base + lm_head
    so we can reuse evaluate_model_mlm() from Cell 40 directly.
    """
    model_base = full_mlm_model.roberta   # same role as model_with_lora
    lm_head    = full_mlm_model.lm_head   # same role as lm_head in your training

    metrics = evaluate_model_mlm(
        model_base     = model_base,
        lm_head        = lm_head,
        dataloader     = dataloader,
        device         = device,
        dataset_config = dataset_config
    )

    return metrics

print("✓ Manual Prompt evaluation wrapper defined")

✓ Manual Prompt evaluation wrapper defined


In [ ]:
def run_manual_prompt_baseline(model_name='sst2', batch_size=8):
    print("=" * 70)
    print(f"MANUAL PROMPT BASELINE  —  {model_name.upper()}")
    print("=" * 70)

    process        = psutil.Process()
    initial_memory = process.memory_info().rss / 1024 / 1024

    # Load tokenized dataset — same path as your training code
    dataset_path = os.path.join(
        SAVE_DIR, 'datasets_tokenized',
        f'{model_name}_{n_shot}shot_tokenized'
    )
    dataset = load_from_disk(dataset_path)

    dataloaders = create_dataloaders({model_name: dataset}, batch_size=batch_size)
    val_loader  = dataloaders[model_name]['validation']
    test_loader = dataloaders[model_name]['test']

    print(f"  Val  batches : {len(val_loader)}")
    print(f"  Test batches : {len(test_loader)}")

    dataset_config = DATASET_CONFIG[model_name]
    print(f"  Pattern      : {dataset_config['pattern']('INPUT_TEXT')}")
    print(f"  Verbalizers  : {dataset_config['verbalizers']}")

    # No training — evaluation only. That IS Manual Prompt.
    start = time.time()

    print("\n— Validation set —")
    val_metrics  = evaluate_manual_prompt(manual_prompt_model, val_loader,  device, dataset_config)

    print("— Test set —")
    test_metrics = evaluate_manual_prompt(manual_prompt_model, test_loader, device, dataset_config)

    elapsed  = time.time() - start
    peak_mem = process.memory_info().rss / 1024 / 1024

    print("\n" + "=" * 70)
    print(f"RESULTS  —  {model_name.upper()}")
    print("=" * 70)
    print(f"  #Params trained : 0")
    print(f"  Test Accuracy   : {test_metrics['accuracy']*100:.2f}%")
    print(f"  Test F1         : {test_metrics['f1']*100:.2f}%")
    print(f"  Inference time  : {elapsed:.2f}s")
    print(f"  Memory increase : {peak_mem - initial_memory:.2f} MB")

    # Save results to Drive
    save_dir = os.path.join(SAVE_DIR, 'baselines', f'{model_name}_manual_prompt')
    os.makedirs(save_dir, exist_ok=True)

    results = {
        'method'          : 'Manual Prompt',
        'model_name'      : model_name,
        'n_shot'          : n_shot,
        'num_params'      : 0,
        'val_metrics'     : val_metrics,
        'test_metrics'    : test_metrics,
        'inference_time_s': elapsed,
        'peak_memory_mb'  : peak_mem,
    }

    with open(os.path.join(save_dir, 'results.json'), 'w') as f:
        json.dump(results, f, indent=2)

    print(f"✓ Results saved → {save_dir}/results.json")
    return results


# Run on SST-2 first to verify
mp_sst2_results = run_manual_prompt_baseline(model_name='sst2', batch_size=8)

MANUAL PROMPT BASELINE  —  SST2
  Val  batches : 4
  Test batches : 109
  Pattern      : INPUT_TEXT It was <mask>.
  Verbalizers  : {0: 'bad', 1: 'great'}

— Validation set —
  Verbalizer mapping: {10999: 0, 12338: 1}
— Test set —
  Verbalizer mapping: {10999: 0, 12338: 1}

RESULTS  —  SST2
  #Params trained : 0
  Test Accuracy   : 76.72%
  Test F1         : 76.65%
  Inference time  : 16.02s
  Memory increase : 248.10 MB
✓ Results saved → /content/drive/MyDrive/FYP_GradientFree_LoRA/baselines/sst2_manual_prompt/results.json


In [ ]:
datasets_to_run = ['sst2', 'yelp_polarity', 'ag_news', 'dbpedia_14']
all_mp_results  = {}

for ds in datasets_to_run:
    result = run_manual_prompt_baseline(model_name=ds, batch_size=8)
    all_mp_results[ds] = result['test_metrics']
    print()

print("✓ All datasets done")

MANUAL PROMPT BASELINE  —  SST2
  Val  batches : 4
  Test batches : 109
  Pattern      : INPUT_TEXT It was <mask>.
  Verbalizers  : {0: 'bad', 1: 'great'}

— Validation set —
  Verbalizer mapping: {10999: 0, 12338: 1}
— Test set —
  Verbalizer mapping: {10999: 0, 12338: 1}

RESULTS  —  SST2
  #Params trained : 0
  Test Accuracy   : 76.72%
  Test F1         : 76.65%
  Inference time  : 14.69s
  Memory increase : 1.21 MB
✓ Results saved → /content/drive/MyDrive/FYP_GradientFree_LoRA/baselines/sst2_manual_prompt/results.json

MANUAL PROMPT BASELINE  —  YELP_POLARITY
  Val  batches : 4
  Test batches : 4750
  Pattern      : INPUT_TEXT It was <mask>.
  Verbalizers  : {0: 'bad', 1: 'great'}

— Validation set —
  Verbalizer mapping: {10999: 0, 12338: 1}
— Test set —
  Verbalizer mapping: {10999: 0, 12338: 1}

RESULTS  —  YELP_POLARITY
  #Params trained : 0
  Test Accuracy   : 73.88%
  Test F1         : 72.66%
  Inference time  : 693.46s
  Memory increase : 49.31 MB
✓ Results saved → /content/

In [ ]:
# Paper's Manual Prompt numbers from Table II (RoBERTa-Large)
paper_numbers = {
    'sst2'         : 79.82,
    'yelp_polarity': 89.65,
    'ag_news'      : 76.96,
    'dbpedia_14'   : 41.33,
}

print("\n" + "=" * 70)
print("SUMMARY TABLE  —  Manual Prompt")
print("=" * 70)
print(f"{'Dataset':<20} {'Ours (%)':>10} {'Paper (%)':>10} {'Diff':>8}")
print("-" * 50)

accs = []
for ds, m in all_mp_results.items():
    acc   = m['accuracy'] * 100
    paper = paper_numbers.get(ds, '-')
    diff  = f"{acc - paper:+.2f}" if isinstance(paper, float) else '-'
    accs.append(acc)
    print(f"{ds:<20} {acc:>10.2f} {str(paper):>10} {diff:>8}")

print("-" * 50)
print(f"{'Average':<20} {np.mean(accs):>10.2f} {'62.56':>10}")
print("=" * 70)
print("Note: Paper used RoBERTa-Large. Diff is expected — trend matters more.")


SUMMARY TABLE  —  Manual Prompt
Dataset                Ours (%)  Paper (%)     Diff
--------------------------------------------------
sst2                      76.72      79.82    -3.10
yelp_polarity             73.88      89.65   -15.77
ag_news                   67.09      76.96    -9.87
dbpedia_14                31.92      41.33    -9.41
--------------------------------------------------
Average                   62.40      62.56
Note: Paper used RoBERTa-Large. Diff is expected — trend matters more.
